# 00 · Setup

실행 방식 1, 노트북 only 패턴입니다. 모든 학습 코드를 노트북 셀 안에 두고 외부 모듈에 의존하지 않습니다.

이 노트북은 후속 노트북(`01-data_prep`, `02..05`)에서 `%run ./00-setup`으로 불러 변수를 공유합니다.

이 setup에서 정의하는 항목은 다음과 같습니다.

- 패키지 설치 (`mlflow`, `lightning`)
- UC catalog / schema / volume 경로
- 단일 `CONFIG` dict (모델 하이퍼파라미터, batch, epoch)
- MovieLens 25M 다운로드 URL과 데이터 경로
- MLflow experiment 경로
- TorchDistributor child에 넘길 `db_host` / `db_token`

기준 모델·데이터셋은 [`00-foundations/concepts-recommender-baseline.md`](../00-foundations/concepts-recommender-baseline.md), 클러스터 사양은 [`00-foundations/env-databricks-environments.md`](../00-foundations/env-databricks-environments.md)를 참고합니다.

## 패키지 설치

먼저 런타임에 무엇이 들어 있고 무엇을 더 깔아야 하는지 확인합니다. DBR 17.3 LTS ML에 사전 설치된 핵심 버전은 다음과 같습니다.

| 패키지 | 버전 |
|--------|------|
| Python | 3.12 |
| `torch` | 2.7.0 (GPU 클러스터) |
| `torchvision` | 0.22.0 |
| `torchmetrics` | 1.6.0 |
| `mlflow-skinny` | 3.0.1 |
| `transformers` | 4.51.3 |
| `accelerate` | 1.5.2 |
| CUDA | 12.6 |
| cuDNN | 9.5.1 |
| NCCL | 2.26.2 |

다음 두 가지는 추가로 설치합니다.

- `lightning==2.5.1` — Lightning은 DBR 17.3 LTS ML에 포함되어 있지 않아 재현성을 위해 버전을 고정합니다.
- `nvidia-ml-py` — MLflow의 `log_system_metrics=True`가 GPU 메트릭(`utilization`, `memory`, `power`)을 수집할 때 사용하는 NVIDIA Management Library 바인딩입니다. 없으면 MLflow는 "Skip logging GPU metrics" 로그만 찍고 GPU 차트가 비어 있게 됩니다 (학습 자체는 정상입니다).

In [ ]:
%pip install --quiet "lightning==2.5.1" "nvidia-ml-py"
%restart_python

## UC 경로

자기 워크스페이스에 맞춰 `CATALOG`, `SCHEMA`, `VOLUME`을 수정합니다.

In [ ]:
CATALOG = "main"
SCHEMA = "distributed_cookbook"
VOLUME = "dataset"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATA_DIR = f"{VOLUME_ROOT}/ml-25m"      # train/, val/, _meta.json 위치
CKPT_DIR = f"{VOLUME_ROOT}/checkpoints"

ML25M_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"VOLUME_ROOT={VOLUME_ROOT}")
print(f"DATA_DIR={DATA_DIR}")
print(f"CKPT_DIR={CKPT_DIR}")

## MLflow experiment

워크스페이스 사용자 폴더 아래에 experiment를 만듭니다. 모든 토폴로지(1×1 / 1×N / M×N)는 같은 experiment에 run으로 누적됩니다.

In [ ]:
import mlflow

USERNAME = (
    spark.sql("SELECT current_user() AS user").first()["user"]
)
EXPERIMENT_PATH = f"/Users/{USERNAME}/recommender-notebook-based"

# 1×1 / 1×N / M×N의 모든 토폴로지 run이 하나의 experiment에 누적되도록 명시적으로 생성한다.
experiment = mlflow.get_experiment_by_name(EXPERIMENT_PATH)
if experiment is None:
    EXPERIMENT_ID = mlflow.create_experiment(EXPERIMENT_PATH)
else:
    EXPERIMENT_ID = experiment.experiment_id
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
print(f"EXPERIMENT_PATH={EXPERIMENT_PATH} (id={EXPERIMENT_ID})")

## 단일 CONFIG

ML-25M 단일 데이터셋에 맞춘 단일 하이퍼파라미터입니다. 1×1 / 1×N / M×N 토폴로지는 모델·데이터를 **공유**하고 launcher 설정과 world_size만 다르게 가져갑니다. global batch는 `batch_size_per_gpu × world_size`로 자연스럽게 확장됩니다.

학습 루프는 epoch 기반이고 `EarlyStopping(patience=3, min_delta=1e-4)`이 실제 종료 시점을 결정합니다. `num_epochs`는 상한, `max_steps_per_epoch`는 epoch당 step 캡으로 15분 budget을 보호합니다.

`n_users` / `n_items`는 `01-data_prep` 실행 결과로 `DATA_DIR/_meta.json`에 기록됩니다. 이 setup 노트북은 metadata가 존재하면 자동으로 불러 `CONFIG`에 주입합니다.

각 키의 의미는 다음 표와 같습니다.

| 키 | 값 | 비고 |
|----|-----|-----|
| `emb_dim` | 64 | 두 tower 공통 |
| `tower_hidden` | (256, 128) | |
| `batch_size_per_gpu` | 4096 | DDP에서 per-rank, global은 ×world_size |
| `num_epochs` | 10 | 상한 |
| `max_steps_per_epoch` | 200 | epoch당 step 캡 |
| `val_ratio` | 0.1 | data_prep에서 적용 |
| `n_users` | metadata에서 로드 | ML-25M 기준 ≈ 162,541 |
| `n_items` | metadata에서 로드 | ML-25M 기준 ≈ 59,047 |

In [ ]:
import json

CONFIG = {
    "emb_dim": 64,
    "tower_hidden": (256, 128),
    "batch_size_per_gpu": 4096,
    "num_epochs": 10,
    "max_steps_per_epoch": 200,
    "val_ratio": 0.1,
    # n_users / n_items는 data_prep 결과로 채워진다.
    "n_users": None,
    "n_items": None,
}

# val_loss 기준 early stopping. 모든 토폴로지 공통.
PATIENCE = 3
MIN_DELTA = 1e-4

meta_path = os.path.join(DATA_DIR, "_meta.json")
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    CONFIG["n_users"] = meta["n_users"]
    CONFIG["n_items"] = meta["n_items"]
    print(f"loaded metadata: n_users={meta['n_users']:,} n_items={meta['n_items']:,} train_rows={meta.get('n_train_rows', 'n/a'):,} val_rows={meta.get('n_val_rows', 'n/a'):,}")
else:
    print(f"[warn] {meta_path} not found. Run 01-data_prep first before training.")

for k, v in CONFIG.items():
    print(f"  CONFIG[{k!r}] = {v}")
print(f"early stopping: patience={PATIENCE} min_delta={MIN_DELTA}")

## TorchDistributor child용 자격증명

TorchDistributor가 띄운 worker는 driver의 Databricks 자격증명을 자동으로 상속하지 않습니다. driver에서 host/token을 한 번 읽어 두고, 학습 함수의 인자로 명시 전달합니다 ([debug-common-pitfalls.md §2-1](../00-foundations/debug-common-pitfalls.md)).

In [ ]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DB_HOST = ctx.extraContext().apply("api_url")
DB_TOKEN = ctx.apiToken().get()
print(f"DB_HOST={DB_HOST}")